# Attendance Grading (Live Only)

* Step 1: Download an updated version of the gradebook
* Step 2: Input the proper session length into the session_length variable
* Step 3: Input the live attendance csv name into the live_csv_name variable
* Step 4: Input the assignment name from the gradebook into the assignment_name variable

Results are retrieved in "grading_results.csv". The remaining csv's are used for debugging purposes. "Testing.csv" returns with more columns joined to the gradebook: "Name", "Total duration (minutes)" and "Grade" to verify that the grading script is placing the correct grades in the correct student's row. "Testing.csv" is for debugging/testing purposes not to be put on Canvas.

## CSV's used for testing/debugging purposes
* Preliminary.csv
* Testing.csv
* grade_count.csv
* grade_verification.csv

## CSV's required for script to function
* Gradebook csv (gradebook_name)
* Zoom live csv (live_csv_name)

## Other necessary inputs
* Zoom session length (session_length)
* Name of assignment in gradebook (assignment_name)

Final results are placed in the "grading_results.csv" use that for inputting into canvas.

In [ ]:
import pandas as pd

# Set the variable session_length to how long the zoom session was for to accurately get grading results (in minutes)
session_length = 170


# Name of live attendance csv (change as needed)
live_csv_name = "zoomus_live_02-02-2026.csv"

# Name of assignment in the gradebook (change as needed)
assignment_name = "2/2/26 Attendance (2598434)"

# Gradebook name (change as needed)
gradebook_name = "2026-02-11T0033_Grades-COP4808_001_13815.csv" 

In [ ]:
# Read the live attendance CSV
live_df = pd.read_csv(live_csv_name)

# Fill in 0's for "< 1" minute values
live_df['Total duration (minutes)'] = live_df['Total duration (minutes)'].apply(lambda x: 0 if x == '< 1' else int(x))

# Sum up the student attendance times grouped by email
live_df = live_df.groupby(["Email"], as_index=False)["Total duration (minutes)"].sum()

live_df

In [ ]:
# Changing Total duration column to int dtype
int_dict = {'Total duration (minutes)': int}
live_df = live_df.astype(int_dict)
print(live_df.dtypes)

In [ ]:
# Implementing grading scale function

def grading_scale(session_length, df):

  ninety_percent_watch = round(session_length * 0.9)
  eighty_percent_watch = round(session_length * 0.8)
  sixty_percent_watch = round(session_length * 0.6)
  forty_percent_watch = round(session_length * 0.4)
  thirty_percent_watch = round(session_length * 0.3)

  df['Grade'] = 'NA'
  df.loc[(df['Total duration (minutes)'] < thirty_percent_watch), 'Grade'] = 0

  df.loc[(df['Total duration (minutes)'] >= thirty_percent_watch) & (df['Total duration (minutes)'] < forty_percent_watch), 'Grade'] = 1

  df.loc[(df['Total duration (minutes)'] >= forty_percent_watch) & (df['Total duration (minutes)'] < sixty_percent_watch), 'Grade'] = 2

  df.loc[(df['Total duration (minutes)'] >= sixty_percent_watch) & (df['Total duration (minutes)'] < eighty_percent_watch), 'Grade'] = 3

  df.loc[(df['Total duration (minutes)'] >= eighty_percent_watch) & (df['Total duration (minutes)'] < ninety_percent_watch), 'Grade'] = 4

  df.loc[(df['Total duration (minutes)'] >= ninety_percent_watch), 'Grade'] = 5

In [ ]:
# Call grading scale with live student attendance and session length

grading_scale(session_length, live_df)

# Used in testing whether the grades were accurately applied to students
live_df.to_csv("./Debugging_csv's/Preliminary.csv")

In [ ]:
# Read in the course gradebook
course_gradebook_df = pd.read_csv(gradebook_name)

# Change zoom live column name from Email to SIS Login ID for joining with gradebook csv
live_df = live_df.rename(columns={"Email": "SIS Login ID"})

live_df_final = live_df.copy()

# Left join the gradebook with the live dataframe (experimental is for debugging)
merged_df_experimental = pd.merge(course_gradebook_df, live_df, on="SIS Login ID", how="left")
merged_df_final = pd.merge(course_gradebook_df, live_df_final, on="SIS Login ID", how="left")

# Create dataframe with students names, emails, watch times, and grade for debugging 
grade_verification_df = pd.concat([merged_df_experimental[["Student", assignment_name]], merged_df_experimental[["SIS Login ID", "Total duration (minutes)", "Grade"]]], axis=1)
grade_verification_df.to_csv("./Debugging_csv's/grade_verification.csv")

# Move Attendance grades into appropriate column at appropriate rows
merged_df_experimental.loc[2:, assignment_name] = merged_df_experimental.loc[2:, "Grade"]
merged_df_final.loc[2:, assignment_name] = merged_df_final.loc[2:, "Grade"]

# Drop unwanted columns from final dataframe
merged_df_final = merged_df_final.drop(["Total duration (minutes)", "Grade"], axis=1)

# Fill in missing values with 0's (missing values means they did not attend live)
merged_df_experimental = merged_df_experimental.fillna(value = {assignment_name: 0})
merged_df_experimental.loc[2:] = merged_df_experimental.loc[2:].fillna(value = {"Grade": 0})
merged_df_final = merged_df_final.fillna(value = {assignment_name: 0})

# CSV where watch times, names, and grades are included to verify the script grades properly
merged_df_experimental.to_csv("./Debugging_csv's/Testing.csv", index=False)

# Final gradebook to submit to Canvas
merged_df_final.to_csv('grading_results.csv', index=False)


In [ ]:
# Gives grade counts for the class
grade_count = merged_df_experimental.copy()
grade_count = grade_count.groupby(['Grade'])['Grade'].size()
grade_count.to_csv('grade_count.csv')

In [ ]:
# Used to compare the gradebook df with the merged_df_final to see if it only changed the assignment name column in the gradebook

column_list = list(course_gradebook_df.columns)
column_list.remove(assignment_name)


comparison_df = merged_df_final.merge(course_gradebook_df, on=column_list, how='outer', suffixes=['', '_'], indicator=True)

comparison_df.to_csv("./Debugging_csv's/comparison.csv")
